In [ ]:
# ======================================================
# Notebook: 6D Hyperparameter Optimisation (CNN surrogate)
# Inputs: (30,6) | Output: (30,)
# Goal: maximise performance
# ======================================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (30,6)
y = np.load("/mnt/data/initial_outputs.npy")     # (30,)

# Normalize targets
y_norm = (y - y.mean()) / (y.std() + 1e-8)

# Reshape for CNN (batch, channel=1, height=6, width=1)
X_cnn = X.reshape(-1,1,6,1)

X_t = torch.tensor(X_cnn, dtype=torch.float32)
y_t = torch.tensor(y_norm.reshape(-1,1), dtype=torch.float32)

# CNN model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(6,1)),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.fc = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = CNN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Train
model.train()
for _ in range(800):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    optimizer.step()

# Candidate sampling
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(6)]
n_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], n_candidates) for b in bounds
])

X_grid_cnn = X_grid.reshape(-1,1,6,1)
X_grid_t = torch.tensor(X_grid_cnn, dtype=torch.float32)

# MC Dropout uncertainty
model.train()
samples = []

with torch.no_grad():
    for _ in range(30):
        samples.append(model(X_grid_t).numpy())

samples = np.stack(samples)
mean_pred = samples.mean(axis=0).flatten()
uncertainty = samples.std(axis=0).flatten()

# Acquisition
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,6)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,6) hyperparameter candidates:")
print(next_points)